# 03 - Train MLP Translation Head

This notebook trains the MLP translation head on top of pre-extracted Concerto features.

Goals:
- use the shared Google Drive structure
- verify that feature files and CLIP label embeddings exist
- train the MLP with `src/train.py`
- save checkpoints and training history back to Drive


### 1. Setup and Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

REPO_DIR = '/content/Deep_learning_project'
BRANCH = 'main'  # change to 'dev' if your team is using the integration branch

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Gandata/Deep_learning_project.git {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch origin
    !git pull origin {BRANCH}

%cd {REPO_DIR}


In [ ]:
!pip install -q -r requirements.txt


### 2. Symlink Drive Folders

We map the shared Drive folders into the repo so every script can use the same paths.

In [ ]:
!rm -f data checkpoints features pretrained
!ln -sf /content/drive/MyDrive/DL_Project/data ./data
!ln -sf /content/drive/MyDrive/DL_Project/checkpoints ./checkpoints
!ln -sf /content/drive/MyDrive/DL_Project/features ./features
!ln -sf /content/drive/MyDrive/DL_Project/pretrained ./pretrained


### 3. Verify Training Inputs

This project expects pre-extracted `.npz` feature files under `features/s3dis_area5/` and label embeddings at `data/s3dis_processed/label_to_clip_embeddings.npy`.

In [ ]:
from pathlib import Path
import numpy as np

features_dir = Path('features/s3dis_area5')
clip_emb_path = Path('data/s3dis_processed/label_to_clip_embeddings.npy')

print('features_dir exists:', features_dir.exists(), features_dir)
print('clip embeddings exist:', clip_emb_path.exists(), clip_emb_path)

feature_files = sorted(features_dir.glob('*.npz')) if features_dir.exists() else []
print('number of feature files:', len(feature_files))
print('sample files:', [p.name for p in feature_files[:5]])

if feature_files:
    sample = np.load(feature_files[0])
    print('sample keys:', sample.files)
    for key in sample.files:
        print(key, sample[key].shape, sample[key].dtype)


### 4. Optional: Generate CLIP Label Embeddings

Run this only if `data/s3dis_processed/label_to_clip_embeddings.npy` is missing.

In [ ]:
from pathlib import Path

clip_emb_path = Path('data/s3dis_processed/label_to_clip_embeddings.npy')
if clip_emb_path.exists():
    print('CLIP label embeddings already exist:', clip_emb_path)
else:
    !python src/clip_utils.py


### 5. Inspect Training Config

In [ ]:
from pathlib import Path
import yaml

config_path = Path('configs/train_mlp_s3dis.yaml')
cfg = yaml.safe_load(config_path.read_text(encoding='utf-8'))
cfg


In [ ]:
# Optional: edit a few config values from inside the notebook, then write back.
# Uncomment and adjust if needed.

# cfg['training']['epochs'] = 50
# cfg['training']['batch_size'] = 4096
# cfg['training']['loss'] = 'mse'
# cfg['training']['resume_from'] = None
# config_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
# print('Updated config written to', config_path)


### 6. Train the Translation Head

This runs the actual training loop implemented in `src/train.py`.

In [ ]:
!python src/train.py --config configs/train_mlp_s3dis.yaml


### 7. Inspect Outputs

The config writes checkpoints into `checkpoints/mlp_s3dis/`, which is symlinked to Drive.

In [ ]:
from pathlib import Path
import json

ckpt_dir = Path('checkpoints/mlp_s3dis')
print('checkpoint dir exists:', ckpt_dir.exists())
print('checkpoint files:')
for path in sorted(ckpt_dir.glob('*')):
    print(' -', path.name)

history_path = ckpt_dir / 'history.json'
if history_path.exists():
    history = json.loads(history_path.read_text(encoding='utf-8'))
    print('\nLast history entries:')
    for row in history[-5:]:
        print(row)
else:
    print('\nNo history.json found yet.')


### 8. Optional: Resume Training Later

In [ ]:
# Example resume workflow.
# 1. Edit the config so training.resume_from points to the checkpoint you want.
# 2. Re-run the training cell above.

# cfg = yaml.safe_load(config_path.read_text(encoding='utf-8'))
# cfg['training']['resume_from'] = 'checkpoints/mlp_s3dis/last_model.pth'
# config_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
# print('Resume checkpoint set.')
